In [29]:
%load_ext autoreload
%autoreload 2
import torch
from utils import goto_project_root
from utils.path_settings import MODEL_SAVE_PATH, DATA_PATH, LOG_PATH, CONFIG_PATH, OBJECT_PATH
from torch.utils.tensorboard import SummaryWriter
import SimulateDatasets.GenTrainingData as g
from utils import create_splits, get_dataloaders, force_remove_dir
from Network_models import Trainer as t
from importlib import reload
import os
reload(t)
reload(g)
import time
import numpy as np
from pytorch3d.transforms import so3_relative_angle
from pytorch3d.transforms import quaternion_to_matrix
import json
import Train_task.train_from_config_names as train
reload(train)
import torchvision.models as models
from torchvision.models import ConvNeXt_Tiny_Weights

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
obj_path_dict = {
    "cow": OBJECT_PATH + "\\cow_mesh\\cow.obj",
    "A1": OBJECT_PATH + "\\stim_simple\\A1_5x6x5_centered.obj",
    "A2": OBJECT_PATH + "\\stim_simple\\A2_5x6x5_centered.obj",
    "B1": OBJECT_PATH + "\\stim_simple\\B1_6x6x6_centered.obj",
    "B2": OBJECT_PATH + "\\stim_simple\\B2_6x6x6_centered.obj",
    "C1": OBJECT_PATH + "\\stim_simple\\C1_5x6x5_centered.obj",
    "C2": OBJECT_PATH + "\\stim_simple\\C2_5x6x5_centered.obj",
    "D1": OBJECT_PATH + "\\stim_simple\\D1_5x6x5_centered.obj",
    "D2": OBJECT_PATH + "\\stim_simple\\D2_5x6x5_centered.obj",
    "E1": OBJECT_PATH + "\\stim_simple\\E1_5x5x6_centered.obj",
    "E2": OBJECT_PATH + "\\stim_simple\\E2_5x5x6_centered.obj",
}
obj_index_dict = {
    "cow": None, 
    "A1": "5x6x5_centered",
    # "A2": "5x6x5_centered",
    # "B1": "6x6x6_centered",
    # "B2": "6x6x6_centered",
    # "C1": "5x6x5_centered",
    # "C2": "5x6x5_centered",
    # "D1": "5x6x5_centered",
    # "D2": "5x6x5_centered",
    # "E1": "5x5x6_centered",
    # "E2": "5x5x6_centered",
}

In [68]:
obj_name = "A1"
obj_index = obj_index_dict.get(obj_name, None)
obj_name = obj_name + "_" + obj_index + ".obj" if obj_index is not None else obj_name + ".obj"
obj_path = obj_path_dict.get(obj_name, None)
network_name = "ConvNeXt_Tiny_" + obj_name.strip(".obj")
FC_size = 16 # This is the input size for the RNN part of the RNN structure (we will be loading FC_RNN).
rnn_hidden_size = 8
task_RNN_pretrained = "0.1q"
rnn_cell_type = "GRU"
rnn_special_name = "constant"
distance_loss = "geodesic_gradual"
seq_len = 100
prep_phase = 50
task_name = "1.1"
re_train = 0

config = {"model_specs": {
    "model_name": "Imported_CNN_RNN",  # to be added (CustomRNN or FC_RNN)
    "model_path": "Network_models.CNN_models",
    "model_params": {
        "convnet": "ConvNeXt_Tiny",
        "weight_string": "IMAGENET1K_V1", 
        "rnn_input_size": FC_size,
        "rnn": {
            "freeze_rnn": False, 
            "model_name": "FC_RNN",
            "model_path": "Network_models.RNN_models",
            "remove_FC": True, 
            "saved_state_dict_path": None,
            "model_params": {
                "input_size": 8,
                "hidden_size": 8, # to be changed
                "num_layers": 1, # to be changed
                "output_size": 3,
                "FC_dim": FC_size,
                "cell_type": rnn_cell_type,  # to be added
            }, 
            "stepwise": False, 
            "saved_model_path": MODEL_SAVE_PATH + 
                                f"\\FC_{FC_size}_{rnn_cell_type}_{1}layer_{rnn_hidden_size}hidden_{task_RNN_pretrained}"
                                f"_model_{'gradual' if distance_loss[-7:] == 'gradual' else ''}"
                                f"{'_' + rnn_special_name if rnn_special_name is not None else ''}\\best_model.pth",
        }, 
    }
}, "model_id": network_name, "device": "cuda", "scheduler": True, # Uses the ReduceLROnPlateau scheduler
    "optimizer_specs": {
    "optimizer_name": "Adam",
    "optimizer_params": {
        "lr": 0.01
    }
}, "distance_loss": distance_loss, "gradual_loss_weighting": "constant+linear", "regularisation_loss": "L2",
    "distance_weight": 1, "output_regs_weight": 1, "silence_activity": False,
    # silence_activity is the knob for suppressing activity in the silence phase
    "training_config": {
        "task_id": "1.1",
        "batch_size": 1024,
        "mini_batch_size": 128,
        "seq_len": seq_len,
        "prep_phase": prep_phase,
        "resolution": 128,
        "cam_position": 8,
        "object_path": obj_path,
        "use_AR": True, 
    }, 'save_path': MODEL_SAVE_PATH + f"\\{network_name}_{task_name}_model", 'log_path': LOG_PATH,
    'check_path': MODEL_SAVE_PATH + f"\\{network_name}_model_checkpoints"}
config['training_config']['data_save_path'] = (DATA_PATH + 
                                               f"\\Data_{task_name}_res{config['training_config']['resolution']}_{obj_name[:-4]}"
                                               f".pth")
    
                                                                                                              

if config['distance_loss'][-7:] == "gradual":
    config['save_path'] = MODEL_SAVE_PATH + (f"\\{network_name}_{task_name}_res{config['training_config']['resolution']}"
                                             f"_model_gradual")
    config['check_path'] = (MODEL_SAVE_PATH + 
                            f"\\{network_name}_res{config['training_config']['resolution']}_model_checkpoints_gradual")
    config['log_path'] = LOG_PATH + (f"\\{network_name}_{task_name}_res{config['training_config']['resolution']}"
                                     f"_model_gradual")
    config['model_specs']['model_params']['stepwise'] = True
else:
    config['model_specs']['model_params']['stepwise'] = False

for path in [config['save_path'], config['log_path'], config['check_path']]:
    if not os.path.exists(path):
        os.makedirs(path)
    elif not re_train:
        print(f"Path already exists for {network_name} in task {task_name}. Assume the model is already trained.")
    else: # re_train
        force_remove_dir(path)
        os.makedirs(path)
print(config['save_path'])
json.dump(config, open(CONFIG_PATH + f"\\{network_name}_{task_name}_res{config['training_config']['resolution']}_configs"
                                     f".json", 
                       "w"))

Path already exists for ConvNeXt_Tiny_A1_5x6x5_centered in task 1.1. Assume the model is already trained.
Path already exists for ConvNeXt_Tiny_A1_5x6x5_centered in task 1.1. Assume the model is already trained.
Path already exists for ConvNeXt_Tiny_A1_5x6x5_centered in task 1.1. Assume the model is already trained.
D:\Projects\mental-rotations\models\ConvNeXt_Tiny_A1_5x6x5_centered_1.1_res128_model_gradual


In [58]:
experimental_trainer = t.Trainer(config)
experimental_trainer.model.resolution = config['training_config']['resolution'] 

C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\torchvision\models\_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


Using a scheduler


In [63]:
# Generate training data for all these training images

reload(g)
training_config = config['training_config']

for stim_name, stim_dimension in obj_index_dict.items():
    stim = f"{stim_name}_{stim_dimension}" if stim_dimension is not None else stim_name
    training_config['object_path'] = obj_path_dict[stim_name]
    training_config['data_save_path'] = (DATA_PATH + f"\\Data_{task_name}_res{training_config['resolution']}_{stim}.pth")
        
    if not os.path.exists(training_config['data_save_path']):
        print('Generating data')
        data = g.gen_training_data(training_config)
# dataloaders = get_dataloaders(data, batch_size = training_config['mini_batch_size'], k = 5)

Generating data
D:\Projects\mental-rotations\data\Data_1.1_res256_cow_train.pth
Overwriting data.


An exception occurred in telemetry logging.Disabling telemetry to prevent further exceptions.
Traceback (most recent call last):
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\file_io.py", line 946, in __log_tmetry_keys
    handler.log_event()
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\event_logger.py", line 97, in log_event
    del self._evt
AttributeError: _evt


consecutive displacements: [0.3680801  0.73579256 1.33302747 2.02184602 2.86125959]


An exception occurred in telemetry logging.Disabling telemetry to prevent further exceptions.
Traceback (most recent call last):
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\file_io.py", line 946, in __log_tmetry_keys
    handler.log_event()
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\event_logger.py", line 97, in log_event
    del self._evt
AttributeError: _evt


D:\Projects\mental-rotations\data\Data_1.1_res256_cow_train2.pth
consecutive displacements: [0.39206612 0.74178189 1.3361536  2.08737197 2.76844963]


...

Generating data
D:\Projects\mental-rotations\data\Data_1.1_res256_A1_5x6x5_centered_train.pth
consecutive displacements: [0.35438767 0.71337469 1.31447894 2.12890248 2.80372803]


An exception occurred in telemetry logging.Disabling telemetry to prevent further exceptions.
Traceback (most recent call last):
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\file_io.py", line 946, in __log_tmetry_keys
    handler.log_event()
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\event_logger.py", line 97, in log_event
    del self._evt
AttributeError: _evt


D:\Projects\mental-rotations\data\Data_1.1_res256_A1_5x6x5_centered_train2.pth
consecutive displacements: [0.36830258 0.73046251 1.34787563 2.07775302 2.78631902]


An exception occurred in telemetry logging.Disabling telemetry to prevent further exceptions.
Traceback (most recent call last):
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\file_io.py", line 946, in __log_tmetry_keys
    handler.log_event()
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\event_logger.py", line 97, in log_event
    del self._evt
AttributeError: _evt


D:\Projects\mental-rotations\data\Data_1.1_res256_A1_5x6x5_centered_test.pth
consecutive displacements: [0.36887128 0.70069196 1.33316557 2.15218932 2.83541493]


In [12]:
data= torch.load(config['training_config']['data_save_path'])
dataloaders = get_dataloaders(data, batch_size = config['training_config']['mini_batch_size'], k = 5)

In [13]:
epochs = 30
print(f"Training {network_name} on task {task_name}, saving to {config['save_path']}; copy the below for logs")
print(f"tensorboard --logdir={config['log_path']}")
for i in range(len(dataloaders)):
    sub_log_path = config["log_path"] + f"\\split_{i + 1}"
    sub_check_path = config["check_path"] + f"\\split_{i + 1}"

    force_remove_dir(sub_log_path) # Probably redundant but helps to ensure no old logs are kept
    os.makedirs(sub_log_path, exist_ok = True)
    os.makedirs(sub_check_path, exist_ok = True)
   
for i, (train_loader, val_loader) in enumerate(dataloaders):
    print(f"Training split {i + 1}")
    experimental_trainer.refresh()
    sub_log_path = config["log_path"] + f"\\split_{i + 1}"
    sub_check_path = config["check_path"] + f"\\split_{i + 1}"

    experimental_trainer.train(train_loader,
                      val_loader,
                      epochs=epochs,
                      save_path = config["save_path"] + f"\\model_{i+1}.pth",
                      check_path= sub_check_path,
                      log_path = sub_log_path)

    experimental_trainer.save_model(config["save_path"] + f"\\model_{i+1}.pth", full=1)
experimental_trainer.load_best_model()
print(f"Training {network_name} on task {task_name} complete. Saving best model to {config['save_path']}.")
experimental_trainer.save_model(config["save_path"] + f"\\best_model.pth", full=1)


Training ConvNeXt_Tiny_A1_5x6x5_centered on task 1.1, saving to D:\Projects\mental-rotations\models\ConvNeXt_Tiny_A1_5x6x5_centered_1.1_res256_model_gradual; copy the below for logs
tensorboard --logdir=D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_A1_5x6x5_centered_1.1_res256_model_gradual
Directory does not exist: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_A1_5x6x5_centered_1.1_res256_model_gradual\split_1
Directory does not exist: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_A1_5x6x5_centered_1.1_res256_model_gradual\split_2
Directory does not exist: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_A1_5x6x5_centered_1.1_res256_model_gradual\split_3
Directory does not exist: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_A1_5x6x5_centered_1.1_res256_model_gradual\split_4
Directory does not exist: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_A1_5x6x5_centered_1.1_res256_model_gradual\split_5
Training split 1
Training split 2
Training split 3
Training split 4
Training spli

In [43]:
task = "1.1"
regenerate = True
network_best_split = {
    "ConvNeXt_Tiny": 3,
}
for network_name, split in network_best_split.items():
    epochs = 200
    trainer = t.Trainer(config)
    trainer.load_checkpoint(config['check_path'] + f"\\split_{split}\\checkpoint29.pth")
    # trainer.scheduler = None
    if (not os.path.exists(config['training_config']['data_save_path'])) or regenerate:
        print('Generating data')
        data = g.gen_training_data(config['training_config'])
    import utils as u
    reload(u)
    dataloaders = u.get_dataloaders(data, config['training_config']['mini_batch_size'], 1)
    print("Continue training the best model")
    sub_log_path = config['log_path'] + "\\continue_training"
    sub_check_path = config['check_path'] + "\\continue_training"
    force_remove_dir(sub_log_path)
    force_remove_dir(sub_check_path)
    os.makedirs(sub_log_path, exist_ok=True)
    os.makedirs(sub_check_path, exist_ok=True)
    trainer.train(
        dataloaders[0][0],
        dataloaders[0][1],
        epochs=epochs,
        save_path = config['save_path'] + "\\continue_training.pth",
        check_path=sub_check_path,
        log_path = sub_log_path,
    )
    trainer.save_model(config['save_path'] + "\\continue_training.pth", full = 1)
    trainer.load_best_model()
    trainer.save_model(config['save_path'] + "\\best_model.pth", full = 1)

An exception occurred in telemetry logging.Disabling telemetry to prevent further exceptions.
Traceback (most recent call last):
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\file_io.py", line 946, in __log_tmetry_keys
    handler.log_event()
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\event_logger.py", line 97, in log_event
    del self._evt
AttributeError: _evt


Using a scheduler
Generating data
Continue training the best model
Successfully removed directory: D:\Projects\mental-rotations\logs\ConvNeXt_Tiny_A1_5x6x5_centered_1.1_res128_model_gradual\continue_training
Successfully removed directory: D:\Projects\mental-rotations\models\ConvNeXt_Tiny_A1_5x6x5_centered_res128_model_checkpoints_gradual\continue_training


In [44]:
for i, layer in enumerate(experimental_trainer.model.children()):
    if hasattr(layer, "frozen"): 
        print(f"Layer {i} is frozen: {layer.frozen}")
    print(f"Layer {i} is {layer}")
experimental_trainer.refresh()

Layer 0 is frozen: True
Layer 0 is Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
  )
  (1): Sequential(
    (0): CNBlock(
      (block): Sequential(
        (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
        (1): Permute()
        (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
        (3): Linear(in_features=96, out_features=384, bias=True)
        (4): GELU(approximate='none')
        (5): Linear(in_features=384, out_features=96, bias=True)
        (6): Permute()
      )
      (stochastic_depth): StochasticDepth(p=0.0, mode=row)
    )
    (1): CNBlock(
      (block): Sequential(
        (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
        (1): Permute()
        (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
        (3): Linear(in_features=96, out_features=384, bias=True)
  

In [35]:
experimental_trainer.load_model(config["save_path"] + f"\\best_model_state_dict.pth", full=0)
experimental_trainer.model.object_path = config['training_config']['object_path']
experimental_trainer.model.cam_position = config['training_config']['cam_position']
experimental_trainer.model.rotate(torch.tensor([1, 0, 0, 0], dtype = torch.get_default_dtype(), device = experimental_trainer.device))

An exception occurred in telemetry logging.Disabling telemetry to prevent further exceptions.
Traceback (most recent call last):
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\file_io.py", line 946, in __log_tmetry_keys
    handler.log_event()
  File "C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\iopath\common\event_logger.py", line 97, in log_event
    del self._evt
AttributeError: _evt


object path C:\Users\timmy\Documents\Projects_dir\Mental_Rotations\mental-rotations\data\stim_simple\A1_5x6x5_centered.obj
Cam position 8


tensor([[ 9.7789e-01,  1.8294e-01, -5.6185e-02,  8.4260e-02],
        [ 9.9275e-01,  1.0002e-01, -3.0292e-02,  5.9374e-02],
        [ 9.9812e-01,  4.9181e-02, -1.3541e-02,  3.3962e-02],
        [ 9.9955e-01,  2.2875e-02, -5.6313e-03,  1.8441e-02],
        [ 9.9989e-01,  1.0463e-02, -1.9858e-03,  1.0100e-02],
        [ 9.9997e-01,  4.9533e-03, -2.9753e-04,  5.7783e-03],
        [ 9.9999e-01,  2.6181e-03,  4.8329e-04,  3.5142e-03],
        [ 1.0000e+00,  1.6577e-03,  8.4172e-04,  2.2670e-03],
        [ 1.0000e+00,  1.2636e-03,  1.0035e-03,  1.5287e-03],
        [ 1.0000e+00,  1.0947e-03,  1.0733e-03,  1.0616e-03],
        [ 1.0000e+00,  1.0150e-03,  1.1001e-03,  7.5354e-04],
        [ 1.0000e+00,  9.7260e-04,  1.1069e-03,  5.4745e-04],
        [ 1.0000e+00,  9.4765e-04,  1.1051e-03,  4.1014e-04],
        [ 1.0000e+00,  9.3215e-04,  1.1001e-03,  3.1992e-04],
        [ 1.0000e+00,  9.2228e-04,  1.0945e-03,  2.6166e-04],
        [ 1.0000e+00,  9.1597e-04,  1.0896e-03,  2.2474e-04],
        

In [36]:
experimental_trainer.save_model(config['save_path'] + "\\best_model.pth", full = 1)

In [61]:
experimental_trainer.model.train()

Imported_CNN_RNN(
  (conv): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_featu